# Tutorial 1 — Prepare brain IDPs

**Goal:** create label-aligned predictor (`X`) and outcome (`Y`) tables. The external input is a BN region-by-IDP table; HomoloMap supplies the cell-type predictors. This notebook performs only data preparation and quality control.

<!-- github-visual-preview -->
<table><tr>
<td><img src="figures/01_brain_idp_surface.png" alt="Example brain IDP on the BN cortical surface"></td>
<td><img src="figures/01_cell_map_surface.png" alt="Example HomoloMap cell map on the BN cortical surface"></td>
</tr></table>

*Deterministic surfplot previews generated in the common BN space. Running the notebook redraws both maps from the current IDP and selected cell type.*


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

HERE = Path.cwd() if (Path.cwd() / 'tutorial_utils.py').exists() else Path.cwd() / 'tutorials'
sys.path.insert(0, str(HERE))
from tutorial_utils import find_repo_root, load_celltype_map, load_idps, align_and_validate

ROOT = find_repo_root()
CELLTYPE_LEVEL = 'subclass'  # or 'cluster'
# Series, DataFrame, CSV/GIFTI/NIfTI path, or loaded nibabel image.
IDP_INPUT = None  # e.g. ROOT / 'my_data' / 'brain_idps_bn.csv'
IDP_SPACE = 'fslr'
IDP_ATLAS = 'BN'
IDP_ATLAS_PATH = None  # required when the source atlas is not cached

## Input contract

`HomoloMap.transforms.load_data` accepts a regional `Series`/`DataFrame`, a CSV path, a scalar GIFTI or NIfTI path, or a loaded nibabel image. It converts each input into the same ROI-by-IDP `DataFrame`. For regional tables, the index contains atlas labels; for images, specify the source space and matching parcellation. NIfTI support requires `pip install -e ".[volume]"`. Labels—not row position—define alignment.

Examples: `IDP_INPUT = dataframe`, `IDP_INPUT = Path('idps.csv')`, `IDP_INPUT = Path('surface.func.gii')`, or `IDP_INPUT = nib.load('effect.nii.gz')`. The fallback below creates deterministic toy IDPs only to verify execution.

In [ ]:
from HomoloMap.transforms import load_data

X_source = load_celltype_map(CELLTYPE_LEVEL, ROOT)
if IDP_INPUT is None:
    Y_source, is_toy = load_idps(None, X_source.index, seed=42)
else:
    Y_source = load_data(
        IDP_INPUT, space=IDP_SPACE, atlas=IDP_ATLAS,
        path=IDP_ATLAS_PATH, trg='BN', smooth=False,
    )
    is_toy = False
X, Y = align_and_validate(X_source, Y_source)

print('Toy IDPs:', is_toy)
print('Cell types:', X.shape, 'IDPs:', Y.shape)
print('BN labels:', X.index.min(), 'to', X.index.max())
display(Y.head())

In [ ]:
from HomoloMap.plotting import plot_left

idp_name = Y.columns[0]
celltype_name = 'Sst' if 'Sst' in X.columns else X.columns[0]

fig_idp = plot_left(
    Y[idp_name], atlas='BN', surf='inflated', cmap='coolwarm',
    cbar_label='IDP value', title=f'Brain IDP: {idp_name}',
    title_fontsize=11, figsize=(8, 2.7), render_scale=(2, 2),
)
fig_cellmap = plot_left(
    X[celltype_name], atlas='BN', surf='inflated', cmap='YlOrRd',
    cbar_label='Mapped proportion', title=f'Cell map: {celltype_name}',
    title_fontsize=11, figsize=(8, 2.7), render_scale=(2, 2),
)

## Save validated inputs

Later tutorials load these files. Record the atlas, hemisphere, feature resolution, retained ROI labels, and mapping audit with a scientific analysis.

In [ ]:
OUTPUT = ROOT / 'tutorial_outputs'
OUTPUT.mkdir(exist_ok=True)
X.to_csv(OUTPUT / 'aligned_celltype_predictors.csv')
Y.to_csv(OUTPUT / 'aligned_brain_idps.csv')
print('Saved to', OUTPUT.resolve())

<!-- tutorial-visual-summary -->
### Visual quality control
The upper panel shows the supplied brain IDPs across ordered BN labels; the lower panel confirms compositional closure.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
Y.plot(ax=axes[0], linewidth=1.5)
axes[0].set(ylabel='IDP value', title='Brain IDPs in BN regional order')
axes[0].legend(frameon=False, ncol=min(4, Y.shape[1]))
axes[1].plot(X.index, X.sum(axis=1), color='#2a9d8f', linewidth=1.5)
axes[1].axhline(1, color='0.25', linestyle='--', linewidth=0.8)
axes[1].set(xlabel='BN region label', ylabel='Row sum', ylim=(0.98, 1.02))
sns.despine()
fig.tight_layout()
